# Scoring de riesgo — Parte 2

Risk score por usuario a partir de las 5 señales confirmadas en el EDA (H1–H5).

**Enfoque:** score heurístico (suma ponderada de señales) como motor — explicable, produce las `top_signals` directamente. Un Isolation Forest valida el ranking de forma independiente.

**Pasos:** (1) tabla de features → (2) volumen relativo a los pares → (3) score 0–100 → (4) categoría → (5) top signals + validación.

In [1]:
import pandas as pd
import numpy as np

DATA = "../data/"
users = pd.read_csv(DATA + "user_inventory.csv",       parse_dates=["created_at"])
perms = pd.read_csv(DATA + "permission_inventory.csv", parse_dates=["assigned_at", "expires_at"])
logs  = pd.read_csv(DATA + "access_logs.csv",          parse_dates=["timestamp"])

## Paso 1 — Tabla de features (una fila por usuario)

Cada señal del EDA se traduce a una columna, con los mismos cruces ya validados:

| Columna | Señal | Cómo se calcula |
|---|---|---|
| `volumen`         | H3 | número de accesos del usuario |
| `n_sin_permiso`   | H1 | accesos a (usuario, recurso) que el usuario **no** tiene asignado |
| `pct_off_hours`   | H4 | fracción de accesos fuera de 7–20 h |
| `n_expirados`     | H5 | accesos usando un permiso ya vencido |
| `inactiva_activa` | H2 | flag: cuenta `Inactive` con actividad |

> **Nota sobre H2 (cuenta inactiva activa):** los datos no incluyen fecha de baja (`status` es el estado actual, sin historia), por lo que no puede confirmarse que el acceso sea posterior a la desactivación. La señal se interpreta como *"cuenta actualmente inactiva con actividad reciente"* — un hallazgo válido (acceso post-baja u offboarding incompleto), pero con **peso medio** por la ambigüedad. Se documenta en Limitaciones (Parte 4).

In [2]:
# Marcas por acceso (mismas reglas que el EDA)
perm_pairs = set(zip(perms.user_id, perms.resource_id))
logs["sin_permiso"] = [(u, r) not in perm_pairs for u, r in zip(logs.user_id, logs.resource_id)]

hora = logs.timestamp.dt.hour
logs["off_hours"] = (hora < 7) | (hora > 20)

# Una fila por usuario con las señales agregables
feat = logs.groupby("user_id").agg(
    volumen       = ("timestamp",   "size"),
    n_sin_permiso = ("sin_permiso", "sum"),
    pct_off_hours = ("off_hours",   "mean"),
).reset_index()

# H5 — accesos con permiso ya expirado, contados por usuario
venc = perms.dropna(subset=["expires_at"])[["user_id", "resource_id", "expires_at"]]
acc_exp = logs.merge(venc, on=["user_id", "resource_id"]).query("timestamp > expires_at")
feat["n_expirados"] = feat.user_id.map(acc_exp.user_id.value_counts()).fillna(0).astype(int)

# H2 — cuenta inactiva con actividad
inactivas = set(users.loc[users.status == "Inactive", "user_id"])
feat["inactiva_activa"] = feat.user_id.isin(inactivas).astype(int)

# department y role para los peer groups (Paso 2)
feat = feat.merge(users[["user_id", "department", "role"]], on="user_id")

print(feat.shape)
feat.sort_values("n_sin_permiso", ascending=False).head(10)

(500, 8)


,user_id,volumen,n_sin_permiso,pct_off_hours,n_expirados,inactiva_activa,department,role
79,USR0080,114,84,0.0,0,0,Engineering,Engineer
59,USR0060,34,27,0.0,0,0,HR,Analyst
39,USR0040,42,25,0.0,0,0,Legal,Admin
40,USR0041,38,25,0.0,0,0,Fintech,Specialist
329,USR0330,35,0,0.0,0,0,Operations,Specialist
331,USR0332,14,0,0.0,0,0,Legal,Analyst
332,USR0333,28,0,0.0,0,0,Operations,Reps
333,USR0334,13,0,0.0,0,0,HR,Analyst
334,USR0335,27,0,0.0,0,0,Operations,Reps
335,USR0336,39,0,0.0,0,0,Data,Reps


## Paso 2 — Volumen relativo a los pares (H3)

El volumen crudo no es comparable entre roles: 50 accesos pueden ser normales para un rol y anómalos para otro. El enunciado pide compararlo contra el peer group (mismo `department` + `role`).

El percentil dentro del grupo solo mide posición, no magnitud: el de mayor volumen de un grupo tranquilo quedaría igual que un outlier de 10×. La señal de riesgo es la **magnitud de la desviación**, no el rango.

Se usa un **z-score robusto**: desvíos típicos por encima de la mediana del grupo.

$$z = \frac{volumen - mediana_{grupo}}{1.4826 \times MAD_{grupo}}$$

- **MAD** (mediana de los desvíos absolutos a la mediana): mide la dispersión sin que los propios outliers la inflen, a diferencia del desvío estándar.
- **1.4826** escala el MAD a unidades comparables con un z-score clásico.
- Los valores negativos se recortan a 0 (solo interesa volumen alto).

**Guardas:** en grupos chicos (<5 usuarios) o con `MAD = 0`, el estadístico intra-grupo no es confiable y se usa la mediana/MAD de toda la población.

In [3]:
MIN_PEERS = 5  # debajo de esto el grupo es muy chico para una dispersión confiable

def robust_z(x, med, mad, mad_fallback):
    """z robusto = (x - mediana) / (1.4826 * MAD); usa el MAD de fallback si el del grupo es 0."""
    escala = 1.4826 * np.where(mad > 0, mad, mad_fallback)
    return ((x - med) / escala).clip(lower=0)  # solo volumen alto

feat["peer_group"] = feat.department + " / " + feat.role
g = feat.groupby("peer_group").volumen

# Estadísticos del grupo (broadcast a cada fila) y de toda la población (fallback)
med, mad_g = g.transform("median"), g.transform(lambda s: (s - s.median()).abs().median())
tam_grupo  = g.transform("size")
med_pob    = feat.volumen.median()
mad_pob    = (feat.volumen - med_pob).abs().median()

# z robusto intra-grupo; en grupos chicos se usa la mediana/MAD global
z_grupo  = robust_z(feat.volumen, med,     mad_g,   mad_pob)
z_global = robust_z(feat.volumen, med_pob, mad_pob, mad_pob)
feat["vol_z"] = z_grupo.where(tam_grupo >= MIN_PEERS, z_global)

feat.sort_values("vol_z", ascending=False)[
    ["user_id", "department", "role", "volumen", "vol_z"]
].head(10)

,user_id,department,role,volumen,vol_z
69,USR0070,Fintech,Analyst,295,49.045114
19,USR0020,Fintech,Reps,396,39.682540
20,USR0021,Fintech,Reps,395,39.570125
39,USR0040,Legal,Admin,42,8.228787
59,USR0060,HR,Analyst,34,6.070417
442,USR0443,Security,Specialist,76,5.171096
79,USR0080,Engineering,Engineer,114,4.571548
312,USR0313,Security,Specialist,67,3.147624
454,USR0455,Data,Reps,54,2.697963
311,USR0312,Legal,Specialist,15,2.697963


## Paso 3 — Score 0–100

Las 5 señales están en escalas distintas (conteo 0–84, z 0–49, % 0–1, flag 0/1) y no son sumables directamente. Dos decisiones:

**1. Normalización a severidad [0,1] con saturación.** Cada señal se divide por un *cap* y se recorta en 1. Pasado el umbral, más no implica más riesgo: acceder a 50 recursos sin permiso no es más concluyente que acceder a 10, ambos son inequívocos. La saturación además evita que un único outlier (USR0080 con 84) comprima al resto a ~0.

| Señal | Cap | Por qué |
|---|---|---|
| `vol_z`         | 3.5 | un z > 3.5 ya está fuera de lo normal (más de 3 sigmas) |
| `n_sin_permiso` | 10  | más de ~10 accesos no autorizados es inequívoco |
| `n_expirados`   | 5   | ídem, en menor escala |
| `pct_off_hours` | —   | ya está en [0,1] |
| `inactiva_activa` | — | ya es 0/1 |

**2. Pesos por severidad** (suman 1, editables). `sin_permiso` es la señal más inequívoca → peso mayor. `inactiva_activa` queda en peso medio por la ambigüedad de H2. Volumen y horario son anomalías de comportamiento que admiten explicación benigna → peso medio. Permisos vencidos: gobernanza, menos grave que el acceso sin permiso.

El score es la suma ponderada × 100. La contribución de cada señal se guarda por separado para explicar el score en el Paso 5.

In [4]:
# Pesos por señal (suman 1) — reflejan severidad de seguridad; editables
PESOS = {
    "sin_permiso":     0.30,  # acceso a recursos no asignados: la señal más inequívoca
    "volumen":         0.20,  # actividad muy por encima de los pares
    "off_hours":       0.20,  # accesos fuera de horario laboral
    "inactiva_activa": 0.15,  # cuenta inactiva con actividad (peso medio: ambigüedad, ver Paso 1)
    "expirados":       0.15,  # uso de permisos vencidos
}

# Caps de saturación: más allá de esto "ya es claramente anómalo, más no suma"
CAP_Z, CAP_PERM, CAP_EXP = 3.5, 10, 5

# Severidad [0,1] de cada señal
sev = pd.DataFrame({
    "sin_permiso":     (feat.n_sin_permiso / CAP_PERM).clip(upper=1),
    "volumen":         (feat.vol_z         / CAP_Z   ).clip(upper=1),
    "off_hours":        feat.pct_off_hours.clip(upper=1),
    "inactiva_activa":  feat.inactiva_activa,
    "expirados":       (feat.n_expirados   / CAP_EXP ).clip(upper=1),
})

# Contribución de cada señal al score (guardada para las top_signals del Paso 5)
contrib = sev * pd.Series(PESOS)
feat["score"] = (contrib.sum(axis=1) * 100).round(1)

feat.sort_values("score", ascending=False)[
    ["user_id", "score", "n_sin_permiso", "vol_z", "pct_off_hours", "inactiva_activa", "n_expirados"]
].head(15)

,user_id,score,n_sin_permiso,vol_z,pct_off_hours,inactiva_activa,n_expirados
39,USR0040,50.0,25,8.228787,0.000000,0,0
59,USR0060,50.0,27,6.070417,0.000000,0,0
79,USR0080,50.0,84,4.571548,0.000000,0,0
49,USR0050,32.4,0,1.410299,0.466667,0,45
40,USR0041,30.0,25,0.000000,0.000000,0,0
69,USR0070,28.2,0,49.045114,0.410169,0,0
11,USR0012,26.2,0,0.000000,0.562500,1,0
29,USR0030,25.8,0,1.011736,1.000000,0,0
9,USR0010,25.0,0,0.000000,0.500000,1,0
10,USR0011,24.5,0,0.000000,0.476190,1,0


## Paso 4 — Categoría de riesgo

El enunciado pide una categoría `LOW / MEDIUM / HIGH / VERY_HIGH`. La distribución condiciona el método de corte:

- 261 de 500 usuarios (52%) tienen score = 0 (sin señales).
- Los 239 restantes forman una cola larga con pocos casos despegados arriba.

Un corte por percentiles no aplica: con más de la mitad en 0, todo percentil hasta el ~52 cae dentro de la masa de ceros. El score, en cambio, es interpretable en puntos (`sin_permiso` saturado = 30, una señal de comportamiento = 20), lo que permite cortar por su significado:

| Categoría | Corte | Qué representa |
|---|---|---|
| `VERY_HIGH` | ≥ 30 | la señal más fuerte completa, o varias combinadas |
| `HIGH`      | 20–30 | una señal fuerte saturada |
| `MEDIUM`    | 10–20 | una señal moderada activa |
| `LOW`       | < 10  | limpio o ruido menor |

**Trade-off:** los cortes son fijos (política de seguridad), no se adaptan a la población. En producción se recalibrarían sobre datos históricos; quedan como constante editable.

In [5]:
# Cortes de categoría (puntos de score) — editables según política de seguridad
CORTES = [0, 10, 20, 30, 100]
ETIQUETAS = ["LOW", "MEDIUM", "HIGH", "VERY_HIGH"]

feat["category"] = pd.cut(
    feat.score, bins=CORTES, labels=ETIQUETAS,
    right=False, include_lowest=True,
)

# Distribución resultante (de mayor a menor riesgo)
feat.category.value_counts().reindex(ETIQUETAS[::-1])

category
VERY_HIGH      5
HIGH           9
MEDIUM        18
LOW          468
Name: count, dtype: int64

## Paso 5 — Top signals + validación

**a) Top signals.** Las señales que más aportaron al score de cada usuario, a partir de `contrib` (Paso 3) y traducidas a texto. Es la explicación que consume la API de la Parte 3.

**b) Validación.**
1. **Por evidencia de comportamiento**, no por ID redondo: los usuarios top deben tener señales reales que justifiquen el score (las propias `top_signals`).
2. **Isolation Forest** como control independiente del ranking: no usa los pesos ni los cortes, solo la geometría de las 5 features. Su coincidencia con la heurística indica que el resultado no depende de la calibración manual.

In [6]:
# Texto legible por señal (usa los valores reales del usuario)
TEXTOS = {
    "sin_permiso":     lambda r: f"Accede a {int(r.n_sin_permiso)} recursos sin permiso asignado",
    "volumen":         lambda r: f"Volumen {r.vol_z:.0f}σ por encima de su peer group",
    "off_hours":       lambda r: f"{r.pct_off_hours:.0%} de accesos fuera de horario",
    "inactiva_activa": lambda r: "Cuenta inactiva con actividad reciente",
    "expirados":       lambda r: f"{int(r.n_expirados)} accesos con permiso vencido",
}

def top_signals(i, n=3):
    """Las n señales que más aportaron al score del usuario i, como texto."""
    aportes = contrib.loc[i]
    señales = aportes[aportes > 0].sort_values(ascending=False).index[:n]
    return [TEXTOS[s](feat.loc[i]) for s in señales]

feat["top_signals"] = [top_signals(i) for i in feat.index]

# Entregable: usuarios ordenados por riesgo con su explicación
salida = feat.sort_values("score", ascending=False)[["user_id", "score", "category", "top_signals"]]
salida.head(14).to_dict("records")

[{'user_id': 'USR0040',
  'score': 50.0,
  'category': 'VERY_HIGH',
  'top_signals': ['Accede a 25 recursos sin permiso asignado',
   'Volumen 8σ por encima de su peer group']},
 {'user_id': 'USR0060',
  'score': 50.0,
  'category': 'VERY_HIGH',
  'top_signals': ['Accede a 27 recursos sin permiso asignado',
   'Volumen 6σ por encima de su peer group']},
 {'user_id': 'USR0080',
  'score': 50.0,
  'category': 'VERY_HIGH',
  'top_signals': ['Accede a 84 recursos sin permiso asignado',
   'Volumen 5σ por encima de su peer group']},
 {'user_id': 'USR0050',
  'score': 32.4,
  'category': 'VERY_HIGH',
  'top_signals': ['45 accesos con permiso vencido',
   '47% de accesos fuera de horario',
   'Volumen 1σ por encima de su peer group']},
 {'user_id': 'USR0041',
  'score': 30.0,
  'category': 'VERY_HIGH',
  'top_signals': ['Accede a 25 recursos sin permiso asignado']},
 {'user_id': 'USR0070',
  'score': 28.2,
  'category': 'HIGH',
  'top_signals': ['Volumen 49σ por encima de su peer group',
   '

In [7]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Control independiente: las 5 features crudas, estandarizadas (sin los pesos ni los caps de la heurística)
X = feat[["n_sin_permiso", "vol_z", "pct_off_hours", "inactiva_activa", "n_expirados"]]
Xs = StandardScaler().fit_transform(X)

iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42)
iso.fit(Xs)
feat["if_score"] = -iso.score_samples(Xs)  # mayor = más anómalo

# Coincidencia del ranking del IF con el de la heurística
K = 14  # tamaño de HIGH + VERY_HIGH
heur_top = set(feat.nlargest(K, "score").user_id)
if_top   = set(feat.nlargest(K, "if_score").user_id)
rho = feat[["score", "if_score"]].corr(method="spearman").iloc[0, 1]

print(f"Coincidencia en el top-{K}:        {len(heur_top & if_top)}/{K}")
print(f"Correlación de rankings (Spearman): {rho:.2f}")
print(f"Marcados por IF pero NO por la heurística: {sorted(if_top - heur_top)}")

Coincidencia en el top-14:        14/14
Correlación de rankings (Spearman): 0.96
Marcados por IF pero NO por la heurística: []
